# Fase 2 — Modelos ML: LR y RF con TF-IDF

Se conservan **Logistic Regression (LR)** y **Random Forest (RF)** y se elimina GloVe y SVM.

Flujo experimental:

1. Sobre el **train oficial**, se ejecuta Nested CV 5×5 para cada combinación `variante × preprocessing × modelo`.
2. Por cada variante se selecciona el mejor preprocessing de **LR** y el mejor preprocessing de **RF** usando únicamente `Outer F1-Macro`.
3. Solo esas 6 configuraciones finales (3 variantes × 2 modelos) se reentrenan sobre todo el train y se evalúan una única vez en el **test oficial**.

El test oficial no participa en la selección de preprocessing ni de hiperparámetros.

## 1. Importación de librerías

In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, accuracy_score,
    precision_score, recall_score,
    classification_report, confusion_matrix
)


## 2. Configuración

In [2]:
DATA_DIR = "../data"
PREPS = ["normal", "stem", "lemma"]
VARIANTES = ["mx", "es", "cu"]
MODELOS = ["LR", "RF"]

FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun"
]

STOP_WORDS = stopwords.words("spanish")
RANDOM_STATE = 42

CV_OUTER = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
CV_INNER = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Preprocesamientos : {PREPS}")
print(f"Modelos            : {MODELOS}")
print("Representación     : TF-IDF")
print(f"Outer CV folds     : {CV_OUTER.n_splits}")
print(f"Inner CV folds     : {CV_INNER.n_splits}")


Preprocesamientos : ['normal', 'stem', 'lemma']
Modelos            : ['LR', 'RF']
Representación     : TF-IDF
Outer CV folds     : 5
Inner CV folds     : 5


## 3. Pipelines TF-IDF y GridSearch

In [3]:
def build_tfidf_prep():
    return ColumnTransformer([
        ("tfidf_word", TfidfVectorizer(
            stop_words=STOP_WORDS,
            max_features=10000
        ), "MESSAGE_CLEAN"),
        ("ling", "passthrough", FEATURE_COLS)
    ])


def build_tfidf_pipelines():
    return {
        "LR": Pipeline([
            ("prep", build_tfidf_prep()),
            ("clf", LogisticRegression(
                max_iter=10000,
                random_state=RANDOM_STATE,
                class_weight="balanced"
            ))
        ]),
        "RF": Pipeline([
            ("prep", build_tfidf_prep()),
            ("clf", RandomForestClassifier(
                random_state=RANDOM_STATE,
                class_weight="balanced",
                n_jobs=1
            ))
        ])
    }


GRIDS_TFIDF = {
    "LR": {
        "prep__tfidf_word__ngram_range": [(1,1), (1,2)],
        "clf__C": [0.1, 1.0, 10],
        "clf__solver": ["liblinear"]
    },
    "RF": {
        "prep__tfidf_word__ngram_range": [(1,1), (1,2)],
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_split": [2, 5]
    }
}

print("Pipelines y grids TF-IDF definidos.")


Pipelines y grids TF-IDF definidos.


## 4. Dimensión TF-IDF por variante y preprocessing

In [4]:
print(f'{"Variante":10} {"Preprocesamiento":18} {"Dim. TF-IDF":>12} {"Dim. total (+9)":>16}')
print("-" * 60)

for prep in PREPS:
    sufijo = "" if prep == "normal" else f"_{prep}"
    for ds in VARIANTES:
        df_tr = pd.read_csv(f"{DATA_DIR}/train_clean{sufijo}_{ds}.csv")
        X = df_tr[["MESSAGE_CLEAN"] + FEATURE_COLS]
        Xt = build_tfidf_prep().fit_transform(X)
        dim_tfidf = Xt.shape[1] - len(FEATURE_COLS)
        print(f"{ds:10} {prep:18} {dim_tfidf:>12,} {Xt.shape[1]:>16,}")


Variante   Preprocesamiento    Dim. TF-IDF  Dim. total (+9)
------------------------------------------------------------
mx         normal                    8,170            8,179
es         normal                    7,939            7,948
cu         normal                    9,431            9,440
mx         stem                      4,953            4,962
es         stem                      5,143            5,152
cu         stem                      5,602            5,611
mx         lemma                     6,300            6,309
es         lemma                     6,303            6,312
cu         lemma                     7,232            7,241


## 5. Nested CV sobre TRAIN

Aquí **no se carga el test oficial**. Cada preprocessing se evalúa por separado para LR y RF. El criterio para escoger el preprocessing final es el `Outer F1 mean`.

In [5]:
RESULTADOS_CV = []


def run_nested_cv(prep, dataset_nombre, df_tr):
    X_tr = df_tr[["MESSAGE_CLEAN"] + FEATURE_COLS].copy()
    y_tr = df_tr["IS_IRONIC"].astype(int).to_numpy()

    print(f'\n{"="*68}')
    print(f"PREP={prep.upper()} | VARIANTE={dataset_nombre.upper()} | TF-IDF")
    print(f'{"="*68}')

    for modelo_nombre in MODELOS:
        grid = GRIDS_TFIDF[modelo_nombre]
        n_cand = int(np.prod([len(v) for v in grid.values()]))
        print(f"\n→ {modelo_nombre} | {n_cand} candidatos | Nested CV...")
        t0 = time.time()

        outer_scores = []
        outer_train_scores = []
        outer_best_params = []

        for fold_idx, (idx_train, idx_val) in enumerate(CV_OUTER.split(X_tr, y_tr), start=1):
            X_ot = X_tr.iloc[idx_train]
            X_ov = X_tr.iloc[idx_val]
            y_ot = y_tr[idx_train]
            y_ov = y_tr[idx_val]

            search = GridSearchCV(
                estimator=build_tfidf_pipelines()[modelo_nombre],
                param_grid=grid,
                cv=CV_INNER,
                scoring="f1_macro",
                n_jobs=-1,
                verbose=0,
                refit=True
            )
            search.fit(X_ot, y_ot)

            pred_train = search.best_estimator_.predict(X_ot)
            pred_outer = search.best_estimator_.predict(X_ov)

            f1_train = f1_score(y_ot, pred_train, average="macro")
            f1_outer = f1_score(y_ov, pred_outer, average="macro")

            outer_train_scores.append(f1_train)
            outer_scores.append(f1_outer)
            outer_best_params.append(search.best_params_)

            print(
                f"   Fold {fold_idx}: Train={f1_train:.4f} | "
                f"Outer={f1_outer:.4f} | Gap={f1_train-f1_outer:.4f}"
            )

        train_mean = float(np.mean(outer_train_scores))
        outer_mean = float(np.mean(outer_scores))
        outer_std = float(np.std(outer_scores, ddof=1))
        gap_mean = train_mean - outer_mean

        RESULTADOS_CV.append({
            "prep": prep,
            "dataset": dataset_nombre,
            "modelo": modelo_nombre,
            "train_f1_mean": train_mean,
            "outer_f1_mean": outer_mean,
            "outer_f1_std": outer_std,
            "gap_mean": gap_mean,
            "outer_scores": outer_scores,
            "outer_train_scores": outer_train_scores,
            "outer_best_params": outer_best_params
        })

        print(
            f"\n   {modelo_nombre} — Outer F1={outer_mean:.4f} ± {outer_std:.4f} | "
            f"Train={train_mean:.4f} | Gap={gap_mean:.4f} | "
            f"Tiempo={time.time()-t0:.1f}s"
        )

print("Función Nested CV lista.")


Función Nested CV lista.


## 6. Ejecutar Nested CV para normal, stem y lemma

In [6]:
for prep in PREPS:
    sufijo = "" if prep == "normal" else f"_{prep}"

    print(f'\n{"#"*72}')
    print(f"PREPROCESAMIENTO: {prep.upper()}")
    print(f'{"#"*72}')

    for ds in VARIANTES:
        # Solo TRAIN. El test todavía no se carga.
        df_train = pd.read_csv(f"{DATA_DIR}/train_clean{sufijo}_{ds}.csv")
        print(f"\n{ds.upper()} | train={len(df_train):,}")
        run_nested_cv(prep, ds, df_train)



########################################################################
PREPROCESAMIENTO: NORMAL
########################################################################

MX | train=2,399

PREP=NORMAL | VARIANTE=MX | TF-IDF

→ LR | 6 candidatos | Nested CV...
   Fold 1: Train=0.9078 | Outer=0.6757 | Gap=0.2321
   Fold 2: Train=0.9027 | Outer=0.6415 | Gap=0.2612
   Fold 3: Train=0.9127 | Outer=0.6721 | Gap=0.2407
   Fold 4: Train=0.9159 | Outer=0.6757 | Gap=0.2402
   Fold 5: Train=0.9111 | Outer=0.6624 | Gap=0.2487

   LR — Outer F1=0.6655 ± 0.0145 | Train=0.9101 | Gap=0.2446 | Tiempo=33.1s

→ RF | 36 candidatos | Nested CV...
   Fold 1: Train=0.8652 | Outer=0.6402 | Gap=0.2250
   Fold 2: Train=0.8455 | Outer=0.6284 | Gap=0.2171
   Fold 3: Train=0.8151 | Outer=0.6519 | Gap=0.1633
   Fold 4: Train=0.8545 | Outer=0.6415 | Gap=0.2130
   Fold 5: Train=0.8005 | Outer=0.5909 | Gap=0.2096

   RF — Outer F1=0.6306 ± 0.0237 | Train=0.8362 | Gap=0.2056 | Tiempo=34.5s

ES | train=2,398

PREP=NOR

## 7. Resultados CV y selección del mejor preprocessing de LR y RF

In [7]:
df_cv = pd.DataFrame([{
    "Prep": r["prep"],
    "Dataset": r["dataset"],
    "Modelo": r["modelo"],
    "Train F1 mean": r["train_f1_mean"],
    "Outer F1 mean": r["outer_f1_mean"],
    "Outer F1 std": r["outer_f1_std"],
    "Gap mean": r["gap_mean"]
} for r in RESULTADOS_CV])

print("RESULTADOS NESTED CV — SOLO TRAIN")
display(
    df_cv.sort_values(
        ["Dataset", "Modelo", "Outer F1 mean"],
        ascending=[True, True, False]
    ).style.format({
        "Train F1 mean": "{:.4f}",
        "Outer F1 mean": "{:.4f}",
        "Outer F1 std": "{:.4f}",
        "Gap mean": "{:.4f}"
    })
)


RESULTADOS NESTED CV — SOLO TRAIN


,Prep,Dataset,Modelo,Train F1 mean,Outer F1 mean,Outer F1 std,Gap mean
10,stem,cu,LR,0.9075,0.6682,0.0302,0.2394
4,normal,cu,LR,0.9455,0.6626,0.0204,0.2829
16,lemma,cu,LR,0.9292,0.6571,0.0326,0.2720
11,stem,cu,RF,0.8593,0.6597,0.0281,0.1996
5,normal,cu,RF,0.8415,0.6586,0.0203,0.1829
17,lemma,cu,RF,0.8358,0.6485,0.0165,0.1872
8,stem,es,LR,0.9224,0.7127,0.0279,0.2097
14,lemma,es,LR,0.9480,0.7087,0.0184,0.2393
2,normal,es,LR,0.9495,0.7071,0.0216,0.2424
15,lemma,es,RF,0.9989,0.7107,0.0234,0.2883


### 7.1 Resultados de cada Outer Fold

Además del promedio del Nested CV, se reporta el resultado individual de cada uno de los 5 folds externos.

Esto permite observar la estabilidad del modelo entre particiones y sustenta el cálculo de la media y la desviación estándar.


In [8]:
# ============================================================
# DETALLE DE LOS 5 OUTER FOLDS
# ============================================================

filas_outer = []

for r in RESULTADOS_CV:
    for fold_idx, (f1_train, f1_outer, params) in enumerate(
        zip(
            r["outer_train_scores"],
            r["outer_scores"],
            r["outer_best_params"],
        ),
        start=1,
    ):
        filas_outer.append({
            "Dataset": r["dataset"],
            "Modelo": r["modelo"],
            "Prep": r["prep"],
            "Outer Fold": fold_idx,
            "F1 Train": f1_train,
            "F1 Outer": f1_outer,
            "Gap": f1_train - f1_outer,
            "Best Params Inner": str(params),
        })


df_outer_detalle = pd.DataFrame(filas_outer)

print("RESULTADOS INDIVIDUALES DE LOS OUTER FOLDS")
print("=" * 100)

display(
    df_outer_detalle.sort_values(
        ["Dataset", "Modelo", "Prep", "Outer Fold"]
    ).style.format({
        "F1 Train": "{:.4f}",
        "F1 Outer": "{:.4f}",
        "Gap": "{:.4f}",
    })
)

df_outer_detalle.to_csv(
    f"{DATA_DIR}/resultados_outer_folds_lr_rf_tfidf.csv",
    index=False,
)

print(
    f"\nDetalle guardado en: "
    f"{DATA_DIR}/resultados_outer_folds_lr_rf_tfidf.csv"
)


RESULTADOS INDIVIDUALES DE LOS OUTER FOLDS


,Dataset,Modelo,Prep,Outer Fold,F1 Train,F1 Outer,Gap,Best Params Inner
80,cu,LR,lemma,1,0.8840,0.6863,0.1977,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
81,cu,LR,lemma,2,0.9936,0.6080,0.3856,"{'clf__C': 10, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
82,cu,LR,lemma,3,0.9889,0.6768,0.3122,"{'clf__C': 10, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
83,cu,LR,lemma,4,0.8886,0.6746,0.2141,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
84,cu,LR,lemma,5,0.8906,0.6399,0.2507,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
20,cu,LR,normal,1,0.9150,0.6881,0.2269,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}"
21,cu,LR,normal,2,0.9162,0.6460,0.2701,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}"
22,cu,LR,normal,3,0.9071,0.6781,0.2290,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
23,cu,LR,normal,4,0.9947,0.6606,0.3342,"{'clf__C': 10, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
24,cu,LR,normal,5,0.9947,0.6403,0.3544,"{'clf__C': 10, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"



Detalle guardado en: ../data/resultados_outer_folds_lr_rf_tfidf.csv


In [9]:
# Una configuración final de LR y una de RF por variante.
# Criterio: mayor Outer F1 mean; empate -> menor Outer F1 std.
MEJORES_CV = (
    df_cv
    .sort_values(
        ["Dataset", "Modelo", "Outer F1 mean", "Outer F1 std"],
        ascending=[True, True, False, True]
    )
    .groupby(["Dataset", "Modelo"], as_index=False)
    .first()
)

print("MEJOR PREPROCESSING POR VARIANTE Y MODELO")
display(
    MEJORES_CV.style.format({
        "Train F1 mean": "{:.4f}",
        "Outer F1 mean": "{:.4f}",
        "Outer F1 std": "{:.4f}",
        "Gap mean": "{:.4f}"
    })
)

assert len(MEJORES_CV) == 6, "Deben existir exactamente 6 configuraciones finales."
print("\nConfiguraciones que pasarán al test:", len(MEJORES_CV))


MEJOR PREPROCESSING POR VARIANTE Y MODELO


,Dataset,Modelo,Prep,Train F1 mean,Outer F1 mean,Outer F1 std,Gap mean
0,cu,LR,stem,0.9075,0.6682,0.0302,0.2394
1,cu,RF,stem,0.8593,0.6597,0.0281,0.1996
2,es,LR,stem,0.9224,0.7127,0.0279,0.2097
3,es,RF,lemma,0.9989,0.7107,0.0234,0.2883
4,mx,LR,normal,0.9101,0.6655,0.0145,0.2446
5,mx,RF,normal,0.8362,0.6306,0.0237,0.2056



Configuraciones que pasarán al test: 6


### 7.2 Outer Folds de las 6 configuraciones seleccionadas

De todas las combinaciones evaluadas en train, se muestran únicamente los 5 resultados externos correspondientes al preprocessing ganador de:

- LR-MX y RF-MX
- LR-ES y RF-ES
- LR-CU y RF-CU

Estas son las 6 configuraciones que posteriormente pasan al test oficial.


In [10]:
# ============================================================
# SOLO LOS OUTER FOLDS DE LAS 6 CONFIGURACIONES GANADORAS
# ============================================================

df_outer_seleccionados = df_outer_detalle.merge(
    MEJORES_CV[["Dataset", "Modelo", "Prep"]],
    on=["Dataset", "Modelo", "Prep"],
    how="inner",
)

print("OUTER FOLDS — CONFIGURACIONES SELECCIONADAS")
print("=" * 100)

display(
    df_outer_seleccionados.sort_values(
        ["Dataset", "Modelo", "Outer Fold"]
    ).style.format({
        "F1 Train": "{:.4f}",
        "F1 Outer": "{:.4f}",
        "Gap": "{:.4f}",
    })
)


# Resumen estadístico de las configuraciones seleccionadas
resumen_seleccionados = (
    df_outer_seleccionados
    .groupby(
        ["Dataset", "Modelo", "Prep"],
        as_index=False,
    )
    .agg(
        F1_Outer_Mean=("F1 Outer", "mean"),
        F1_Outer_Std=("F1 Outer", "std"),
        F1_Outer_Min=("F1 Outer", "min"),
        F1_Outer_Max=("F1 Outer", "max"),
        F1_Train_Mean=("F1 Train", "mean"),
        Gap_Mean=("Gap", "mean"),
    )
)

print("\nRESUMEN FINAL DEL NESTED CV — MEDIA ± DESVIACIÓN ESTÁNDAR")
print("=" * 100)

display(
    resumen_seleccionados.style.format({
        "F1_Outer_Mean": "{:.4f}",
        "F1_Outer_Std": "{:.4f}",
        "F1_Outer_Min": "{:.4f}",
        "F1_Outer_Max": "{:.4f}",
        "F1_Train_Mean": "{:.4f}",
        "Gap_Mean": "{:.4f}",
    })
)

print("\nFormato recomendado para reportar:")
for _, row in resumen_seleccionados.iterrows():
    print(
        f"{row['Dataset'].upper()} | "
        f"{row['Modelo']} | "
        f"{row['Prep']} → "
        f"F1-Macro = "
        f"{row['F1_Outer_Mean']:.4f} ± "
        f"{row['F1_Outer_Std']:.4f}"
    )


df_outer_seleccionados.to_csv(
    f"{DATA_DIR}/outer_folds_modelos_seleccionados.csv",
    index=False,
)

resumen_seleccionados.to_csv(
    f"{DATA_DIR}/resumen_nestedcv_modelos_seleccionados.csv",
    index=False,
)


OUTER FOLDS — CONFIGURACIONES SELECCIONADAS


,Dataset,Modelo,Prep,Outer Fold,F1 Train,F1 Outer,Gap,Best Params Inner
15,cu,LR,stem,1,0.8771,0.7103,0.1668,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
16,cu,LR,stem,2,0.9942,0.6430,0.3512,"{'clf__C': 10, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}"
17,cu,LR,stem,3,0.9022,0.6832,0.2191,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}"
18,cu,LR,stem,4,0.8815,0.6683,0.2131,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
19,cu,LR,stem,5,0.8827,0.6361,0.2466,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
20,cu,RF,stem,1,0.8543,0.6994,0.1549,"{'clf__max_depth': 10, 'clf__min_samples_split': 5, 'clf__n_estimators': 300, 'prep__tfidf_word__ngram_range': (1, 2)}"
21,cu,RF,stem,2,0.8331,0.6686,0.1645,"{'clf__max_depth': 10, 'clf__min_samples_split': 5, 'clf__n_estimators': 200, 'prep__tfidf_word__ngram_range': (1, 1)}"
22,cu,RF,stem,3,0.8402,0.6614,0.1788,"{'clf__max_depth': 10, 'clf__min_samples_split': 5, 'clf__n_estimators': 200, 'prep__tfidf_word__ngram_range': (1, 1)}"
23,cu,RF,stem,4,0.8863,0.6456,0.2408,"{'clf__max_depth': 20, 'clf__min_samples_split': 2, 'clf__n_estimators': 200, 'prep__tfidf_word__ngram_range': (1, 1)}"
24,cu,RF,stem,5,0.8824,0.6236,0.2588,"{'clf__max_depth': 20, 'clf__min_samples_split': 5, 'clf__n_estimators': 100, 'prep__tfidf_word__ngram_range': (1, 1)}"



RESUMEN FINAL DEL NESTED CV — MEDIA ± DESVIACIÓN ESTÁNDAR


,Dataset,Modelo,Prep,F1_Outer_Mean,F1_Outer_Std,F1_Outer_Min,F1_Outer_Max,F1_Train_Mean,Gap_Mean
0,cu,LR,stem,0.6682,0.0302,0.6361,0.7103,0.9075,0.2394
1,cu,RF,stem,0.6597,0.0281,0.6236,0.6994,0.8593,0.1996
2,es,LR,stem,0.7127,0.0279,0.6893,0.7466,0.9224,0.2097
3,es,RF,lemma,0.7107,0.0234,0.6826,0.7369,0.9989,0.2883
4,mx,LR,normal,0.6655,0.0145,0.6415,0.6757,0.9101,0.2446
5,mx,RF,normal,0.6306,0.0237,0.5909,0.6519,0.8362,0.2056



Formato recomendado para reportar:
CU | LR | stem → F1-Macro = 0.6682 ± 0.0302
CU | RF | stem → F1-Macro = 0.6597 ± 0.0281
ES | LR | stem → F1-Macro = 0.7127 ± 0.0279
ES | RF | lemma → F1-Macro = 0.7107 ± 0.0234
MX | LR | normal → F1-Macro = 0.6655 ± 0.0145
MX | RF | normal → F1-Macro = 0.6306 ± 0.0237


## 8. Reentrenamiento final y TEST oficial

A partir de aquí recién se carga el test. Para cada una de las 6 configuraciones seleccionadas se realiza un GridSearch final sobre **todo el train** y después se evalúa una sola vez en el test oficial.

In [11]:
RESULTADOS_TEST = []
MODELOS_FINALES = {}
MODELS_DIR = f"{DATA_DIR}/modelos_finales_lr_rf_tfidf"
os.makedirs(MODELS_DIR, exist_ok=True)

for _, sel in MEJORES_CV.iterrows():
    ds = sel["Dataset"]
    modelo_nombre = sel["Modelo"]
    prep = sel["Prep"]
    sufijo = "" if prep == "normal" else f"_{prep}"

    df_train = pd.read_csv(f"{DATA_DIR}/train_clean{sufijo}_{ds}.csv")
    df_test = pd.read_csv(f"{DATA_DIR}/test_clean{sufijo}_{ds}.csv")

    X_train = df_train[["MESSAGE_CLEAN"] + FEATURE_COLS].copy()
    y_train = df_train["IS_IRONIC"].astype(int).to_numpy()
    X_test = df_test[["MESSAGE_CLEAN"] + FEATURE_COLS].copy()
    y_test = df_test["IS_IRONIC"].astype(int).to_numpy()

    print(f'\n{"="*76}')
    print(f"FINAL | {ds.upper()} | {modelo_nombre} | PREP={prep.upper()} | TF-IDF")
    print(f'{"="*76}')

    final_search = GridSearchCV(
        estimator=build_tfidf_pipelines()[modelo_nombre],
        param_grid=GRIDS_TFIDF[modelo_nombre],
        cv=CV_INNER,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0,
        refit=True
    )
    final_search.fit(X_train, y_train)
    estimator = final_search.best_estimator_

    pred_train = estimator.predict(X_train)
    pred_test = estimator.predict(X_test)

    f1_train_final = f1_score(y_train, pred_train, average="macro")
    f1_test = f1_score(y_test, pred_test, average="macro")
    acc_test = accuracy_score(y_test, pred_test)
    p_macro = precision_score(y_test, pred_test, average="macro", zero_division=0)
    r_macro = recall_score(y_test, pred_test, average="macro", zero_division=0)
    p_ir = precision_score(y_test, pred_test, pos_label=1, zero_division=0)
    r_ir = recall_score(y_test, pred_test, pos_label=1, zero_division=0)
    f1_ir = f1_score(y_test, pred_test, pos_label=1, zero_division=0)

    RESULTADOS_TEST.append({
        "dataset": ds,
        "modelo": modelo_nombre,
        "prep": prep,
        "repr": "TF-IDF",
        "outer_f1_mean": float(sel["Outer F1 mean"]),
        "outer_f1_std": float(sel["Outer F1 std"]),
        "gap_cv_mean": float(sel["Gap mean"]),
        "f1_cv_final": float(final_search.best_score_),
        "f1_train_final": float(f1_train_final),
        "f1_macro_test": float(f1_test),
        "accuracy_test": float(acc_test),
        "precision_macro_test": float(p_macro),
        "recall_macro_test": float(r_macro),
        "precision_ironico": float(p_ir),
        "recall_ironico": float(r_ir),
        "f1_ironico": float(f1_ir),
        "best_params": final_search.best_params_,
        "estimator": estimator,
        "y_test": y_test,
        "y_pred_test": pred_test,
        "df_test": df_test.copy()
    })

    MODELOS_FINALES[(ds, modelo_nombre)] = estimator

    print(f"Nested CV seleccionado : {sel['Outer F1 mean']:.4f} ± {sel['Outer F1 std']:.4f}")
    print(f"F1 CV final            : {final_search.best_score_:.4f}")
    print(f"F1 train final         : {f1_train_final:.4f}")
    print(f"F1-Macro TEST          : {f1_test:.4f}")
    print(f"Accuracy TEST          : {acc_test:.4f}")
    print("Mejores parámetros    :", final_search.best_params_)

    joblib.dump(
        estimator,
        f"{MODELS_DIR}/tfidf_{modelo_nombre.lower()}_{ds}_{prep}.pkl"
    )

assert len(RESULTADOS_TEST) == 6
print("\nTotal de evaluaciones sobre test:", len(RESULTADOS_TEST))



FINAL | CU | LR | PREP=STEM | TF-IDF
Nested CV seleccionado : 0.6682 ± 0.0302
F1 CV final            : 0.6756
F1 train final         : 0.8744
F1-Macro TEST          : 0.6718
Accuracy TEST          : 0.7200
Mejores parámetros    : {'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}

FINAL | CU | RF | PREP=STEM | TF-IDF
Nested CV seleccionado : 0.6597 ± 0.0281
F1 CV final            : 0.6695
F1 train final         : 0.8209
F1-Macro TEST          : 0.6796
Accuracy TEST          : 0.7300
Mejores parámetros    : {'clf__max_depth': 10, 'clf__min_samples_split': 5, 'clf__n_estimators': 200, 'prep__tfidf_word__ngram_range': (1, 1)}

FINAL | ES | LR | PREP=STEM | TF-IDF
Nested CV seleccionado : 0.7127 ± 0.0279
F1 CV final            : 0.7120
F1 train final         : 0.8761
F1-Macro TEST          : 0.7179
Accuracy TEST          : 0.7400
Mejores parámetros    : {'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}

FINAL | ES | RF |

## 9. Tabla final — 2 modelos por variante

In [12]:
df_test_final = pd.DataFrame([{
    "Variante": r["dataset"],
    "Modelo": r["modelo"],
    "Prep seleccionado": r["prep"],
    "Outer F1 mean": r["outer_f1_mean"],
    "Outer F1 std": r["outer_f1_std"],
    "F1 CV final": r["f1_cv_final"],
    "F1 train final": r["f1_train_final"],
    "F1-Macro test": r["f1_macro_test"],
    "Accuracy test": r["accuracy_test"],
    "Precision Macro test": r["precision_macro_test"],
    "Recall Macro test": r["recall_macro_test"],
    "F1 Irónico": r["f1_ironico"],
    "Best Params": str(r["best_params"])
} for r in RESULTADOS_TEST])

print("RESULTADOS FINALES — 6 MODELOS")
display(
    df_test_final.sort_values(["Variante", "Modelo"]).style.format({
        "Outer F1 mean": "{:.4f}",
        "Outer F1 std": "{:.4f}",
        "F1 CV final": "{:.4f}",
        "F1 train final": "{:.4f}",
        "F1-Macro test": "{:.4f}",
        "Accuracy test": "{:.4f}",
        "Precision Macro test": "{:.4f}",
        "Recall Macro test": "{:.4f}",
        "F1 Irónico": "{:.4f}"
    })
)

out_csv = f"{DATA_DIR}/resultados_lr_rf_tfidf_final.csv"
df_test_final.to_csv(out_csv, index=False)
print(f"\nCSV guardado: {out_csv}")


RESULTADOS FINALES — 6 MODELOS


,Variante,Modelo,Prep seleccionado,Outer F1 mean,Outer F1 std,F1 CV final,F1 train final,F1-Macro test,Accuracy test,Precision Macro test,Recall Macro test,F1 Irónico,Best Params
0,cu,LR,stem,0.6682,0.0302,0.6756,0.8744,0.6718,0.7200,0.6819,0.6663,0.5459,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 1)}"
1,cu,RF,stem,0.6597,0.0281,0.6695,0.8209,0.6796,0.7300,0.6945,0.6725,0.5525,"{'clf__max_depth': 10, 'clf__min_samples_split': 5, 'clf__n_estimators': 200, 'prep__tfidf_word__ngram_range': (1, 1)}"
2,es,LR,stem,0.7127,0.0279,0.7120,0.8761,0.7179,0.7400,0.7132,0.7275,0.6389,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}"
3,es,RF,lemma,0.7107,0.0234,0.7072,0.9981,0.6883,0.7333,0.6983,0.6825,0.5699,"{'clf__max_depth': None, 'clf__min_samples_split': 5, 'clf__n_estimators': 300, 'prep__tfidf_word__ngram_range': (1, 1)}"
4,mx,LR,normal,0.6655,0.0145,0.6666,0.9007,0.6924,0.7283,0.6932,0.6917,0.5873,"{'clf__C': 1.0, 'clf__solver': 'liblinear', 'prep__tfidf_word__ngram_range': (1, 2)}"
5,mx,RF,normal,0.6306,0.0237,0.6480,0.8410,0.6255,0.6683,0.6256,0.6253,0.4987,"{'clf__max_depth': 20, 'clf__min_samples_split': 5, 'clf__n_estimators': 200, 'prep__tfidf_word__ngram_range': (1, 2)}"



CSV guardado: ../data/resultados_lr_rf_tfidf_final.csv


## 10. Análisis de contribución de características lingüísticas

Se usan únicamente los 6 estimadores finales: LR y RF para cada variante, cada uno con su preprocessing seleccionado.

In [13]:
def extraer_importancia_ling(estimator):
    clf = estimator.named_steps["clf"]

    if hasattr(clf, "feature_importances_"):
        importances = clf.feature_importances_
    elif hasattr(clf, "coef_"):
        coef = clf.coef_
        importances = np.abs(coef[0]) if coef.ndim > 1 else np.abs(coef)
    else:
        return None

    tfidf = estimator.named_steps["prep"].named_transformers_["tfidf_word"]
    n_tfidf = len(tfidf.get_feature_names_out())
    ling = importances[n_tfidf:n_tfidf + len(FEATURE_COLS)]

    if len(ling) != len(FEATURE_COLS):
        return None

    return dict(zip(FEATURE_COLS, ling))


FEAT_ANALYSIS = []
for r in RESULTADOS_TEST:
    imp = extraer_importancia_ling(r["estimator"])
    if imp is not None:
        FEAT_ANALYSIS.append({
            "dataset": r["dataset"],
            "modelo": r["modelo"],
            "prep": r["prep"],
            **imp
        })

df_feat = pd.DataFrame(FEAT_ANALYSIS)
print("IMPORTANCIA DE CARACTERÍSTICAS LINGÜÍSTICAS — MODELOS FINALES")
for ds in VARIANTES:
    print(f"\nVariante: {ds.upper()}")
    sub = df_feat[df_feat["dataset"] == ds].set_index(["modelo", "prep"])[FEATURE_COLS]
    display(sub.style.format("{:.4f}"))

df_feat.to_csv(f"{DATA_DIR}/feature_importance_ling_lr_rf_final.csv", index=False)


IMPORTANCIA DE CARACTERÍSTICAS LINGÜÍSTICAS — MODELOS FINALES

Variante: MX


,,n_exc,n_int,n_may,n_emo,n_ris,n_neg,n_elo,n_com,n_pun
modelo,prep,,,,,,,,,
LR,normal,0.0920,0.0222,0.0428,0.1271,0.5455,0.2184,0.1114,0.1818,0.2376
RF,normal,0.0092,0.0022,0.0041,0.0161,0.0008,0.0101,0.0266,0.0049,0.0091



Variante: ES


,,n_exc,n_int,n_may,n_emo,n_ris,n_neg,n_elo,n_com,n_pun
modelo,prep,,,,,,,,,
LR,stem,0.0830,0.0440,0.0860,0.0445,0.0554,0.2456,0.1550,0.2161,0.4365
RF,lemma,0.0042,0.0055,0.0049,0.0046,0.0003,0.0092,0.0049,0.0079,0.0044



Variante: CU


,,n_exc,n_int,n_may,n_emo,n_ris,n_neg,n_elo,n_com,n_pun
modelo,prep,,,,,,,,,
LR,stem,0.0904,0.0280,0.0121,0.0000,0.9672,0.1073,0.6451,1.4164,0.9632
RF,stem,0.0370,0.0132,0.0196,0.0000,0.0009,0.0048,0.0561,0.0179,0.0036


## 11. Análisis de errores — los 6 modelos finales

In [14]:
for r in RESULTADOS_TEST:
    print(f'\n{"="*76}')
    print(
        f"{r['dataset'].upper()} | {r['modelo']} | "
        f"PREP={r['prep'].upper()} | TF-IDF | "
        f"F1 test={r['f1_macro_test']:.4f}"
    )
    print(f'{"="*76}')

    y_test = r["y_test"]
    y_pred = r["y_pred_test"]

    print(classification_report(
        y_test, y_pred,
        target_names=["No irónico", "Irónico"],
        digits=4,
        zero_division=0
    ))
    print("Matriz de confusión:")
    print(confusion_matrix(y_test, y_pred))

    df_e = r["df_test"].copy()
    df_e["y_pred"] = y_pred
    errores = df_e[df_e["IS_IRONIC"] != df_e["y_pred"]]
    fn = errores[errores["IS_IRONIC"] == 1]
    fp = errores[errores["IS_IRONIC"] == 0]

    print(f"\nFalsos negativos: {len(fn)}")
    for _, row in fn.head(5).iterrows():
        print("  →", str(row["MESSAGE_CLEAN"])[:120])

    print(f"\nFalsos positivos: {len(fp)}")
    for _, row in fp.head(5).iterrows():
        print("  →", str(row["MESSAGE_CLEAN"])[:120])



CU | LR | PREP=STEM | TF-IDF | F1 test=0.6718
              precision    recall  f1-score   support

  No irónico     0.7698    0.8275    0.7976       400
     Irónico     0.5941    0.5050    0.5459       200

    accuracy                         0.7200       600
   macro avg     0.6819    0.6663    0.6718       600
weighted avg     0.7112    0.7200    0.7137       600

Matriz de confusión:
[[331  69]
 [ 99 101]]

Falsos negativos: 99
  → yo pagari los 15 cuc por algo se empiez ahh clar ahor que recuerd no pued porqu no teng telefon fij jajaj
  → por lo vist los partidari de industrial son fiel reflej de ese equip arrog y prepotent sin cultur deport nosotr los guaj
  → la pregunt es los que cre tuv la genial ide per olvid el salari promedi de la socied dond lo van a ofrec o sencill no es
  → rob que cuc cup que validez tien esas moned fuer de nuestr pais mas desastr es un certific autofirm dond cualqu se met y
  → en obtiembr te van a respond esper ahi sent

Falsos positivos: 69
  → q

## 12. Resumen del flujo

```text
TRAIN oficial
   │
   ├── normal
   ├── stem
   └── lemma
        │
        ▼
Nested CV 5×5
LR y RF + TF-IDF
        │
        ▼
Por variante:
mejor preprocessing LR
+
mejor preprocessing RF
        │
        ▼
6 modelos finales
        │
        ▼
GridSearch final sobre todo TRAIN
        │
        ▼
TEST oficial
(solo esos 6 modelos)
```
